# HW1 Starter · 二十分钟跑通第一次 loss 下降

**零环境起步**：本 notebook 在 Colab 免费 T4 上可直接运行，无需本地 GPU。菜单 `代码执行程序 > 更改代码执行程序类型 > T4 GPU`。

> **数据说明**：notebook 的数据与产物只存在于你自己的 Colab 会话中，与本站（localStorage）无关。
> **正式做 HW1** 时请把第 1 节的数据换成你自己的中文语料（见站点 Checklist 第 2 项）。

全篇共 **3 处练习位**（学习率探索 / 采样控制 / 消融雏形）——其余代码已填好，你只管跑通后逐个填空。

## 0 · 环境检查

In [ ]:
import torch, sys
assert sys.version_info >= (3, 8)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} · device = {device}')
# 若 device=cpu：左侧「代码执行程序」→「更改代码执行程序类型」→ 选 T4 GPU

## 1 · 数据：tiny shakespeare（1MB，URL 稳定、分钟级可跑完）

In [ ]:
import urllib.request
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'input.txt')
text = open('input.txt', encoding='utf-8').read()
print(f'{len(text):,} 字符 · 前 80 字：{text[:80]!r}')

## 2 · Tokenizer：minBPE 风格（与站点 BPE 演示同一逻辑）

统计相邻 pair 频次 → 合并最高频者 → 重复。词表 = 基础 256 字节 + 300 次合并。

In [ ]:
def merge(ids, pair, new_id):
    out, j = [], 0
    while j < len(ids):
        if j < len(ids) - 1 and (ids[j], ids[j+1]) == pair:
            out.append(new_id); j += 2
        else:
            out.append(ids[j]); j += 1
    return out

def train_bpe(text, n_merges=300):
    ids = list(text.encode('utf-8'))
    merges = {}                                   # (a, b) -> 新 id
    for i in range(n_merges):
        cnt = {}
        for a, b in zip(ids, ids[1:]):
            cnt[(a, b)] = cnt.get((a, b), 0) + 1
        if not cnt: break
        pair = max(cnt, key=cnt.get)              # 最高频 pair
        merges[pair] = 256 + i
        ids = merge(ids, pair, merges[pair])
    return merges

def encode(text, merges):
    ids = list(text.encode('utf-8'))
    for pair, new_id in sorted(merges.items(), key=lambda kv: kv[1]):  # 按学习顺序重放
        ids = merge(ids, pair, new_id)
    return ids

def decode(ids, merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for pair, new_id in merges.items():
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
    return b''.join(vocab[i] for i in ids).decode('utf-8', errors='replace')

merges = train_bpe(text)
sample = 'First Citizen:'
enc = encode(sample, merges)
print(f'词表大小 {256 + len(merges)} · {sample!r} -> {len(enc)} tokens')
print('可逆性检查:', decode(enc, merges) == sample)

## 3 · 模型：与站点 1.4 参考实现同源的小 GPT

因果注意力 + 残差 + LN + FFN，~1.7M 参数，T4 上每千步约 30 秒。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.register_buffer('mask', torch.tril(torch.ones(256, 256)).view(1, 1, 256, 256))
    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=2)
        q = q.view(B, T, self.n_head, -1).transpose(1, 2)
        k = k.view(B, T, self.n_head, -1).transpose(1, 2)
        v = v.view(B, T, self.n_head, -1).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (k.size(-1) ** 0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, d)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, d, n_head):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d), CausalSelfAttention(d, n_head)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))          # Pre-LN：残差在 LN 外侧
        x = x + self.ffn(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab, d=128, n_head=4, n_layer=3, block=256):
        super().__init__()
        self.block = block
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(block, d)
        self.blocks = nn.ModuleList(Block(d, n_head) for _ in range(n_layer))
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)
    def forward(self, idx, targets=None):
        x = self.emb(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks: x = b(x)
        logits = self.head(self.ln(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

V = 256 + len(merges)
model = TinyGPT(V).to(device)
print(f'{sum(p.numel() for p in model.parameters()):,} 参数')

## 4 · 训练循环（500 步，盯 loss 下降）

In [ ]:
data = torch.tensor(encode(text, merges), dtype=torch.long)
split = int(0.9 * len(data))
def get_batch(split_data, batch=32, block=256):
    ix = torch.randint(len(split_data) - block - 1, (batch,))
    x = torch.stack([split_data[i:i+block] for i in ix])
    y = torch.stack([split_data[i+1:i+block+1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4)   # 练习位 1 会回来动它
for step in range(500):
    x, y = get_batch(data[:split])
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == 499:
        print(f'step {step:4d} · loss {loss.item():.3f}')
# 预期：从 ~5.7（ln V）降到 ~1.8 附近。看清楚了——这就是「训练」的全部。

### TODO(user) 1 · 学习率探索

把 lr 换成 3e-3 和 3e-5 各跑 200 步，记录 loss 曲线。回答：哪个量级崩了、哪个量级太慢？把结论写进你的 HW1 实验记录（checklist 第 7 项）。

In [ ]:
# 练习要求：复制上面的训练循环，分别用 lr=3e-3 与 lr=3e-5 各跑 200 步
# 记录：初始 loss、200 步后 loss、是否出现 NaN
raise NotImplementedError('填完这行再删掉：完成你的两次 lr 实验并记录')

### TODO(user) 2 · temperature / top-k 采样

实现带 temperature 与 top-k 的采样，与站点 Checklist 第 6 项对应。

In [ ]:
@torch.no_grad()
def sample(model, prompt_ids, n_new=200, temperature=1.0, top_k=None):
    # 练习要求：按以下三步补全
    #   1) logits / temperature
    #   2) 若 top_k：只保留前 k 个 logit，其余设 -inf
    #   3) softmax -> 多项式采样 -> 追加进序列，循环 n_new 次
    raise NotImplementedError('实现你的采样函数，然后试试 temperature=0.8, top_k=50')

### TODO(user) 3 · 消融雏形：去掉残差

站点作业卡要求「去残差」消融。这里给最小雏形：把 Block.forward 的两条残差改成直通。

In [ ]:
class BlockNoResidual(nn.Module):
    # 练习要求：复制上面的 Block，把 forward 改为 x = self.attn(self.ln1(x)) 等「无残差」直通
    # 观察：500 步内 loss 是否还能降到 2.5 以下？这就是「残差是深网络生命线」的第一手证据
    raise NotImplementedError('完成你的消融实验，与基线各跑 500 步对比 loss 曲线')

## 下一步

- 回到站点 **HW1 Checklist**，第 5–7 项已经在这里打了地基
- 正式训练把数据换成中文语料，模型放大到 0.1B（本地/AutoDL）
- 三个 TODO 完成后，你的实验记录已覆盖 PA1 验收标准的一半